[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/HyperCoast/blob/main/docs/examples/registry_workflows.ipynb)

# Sensor Registry and Workflow Presets

This notebook demonstrates the registry metadata and coastal workflow presets added to HyperCoast. It uses a small synthetic cube, so it can run without external data downloads.

In [ ]:
# %pip install hypercoast

In [ ]:
import numpy as np
import xarray as xr

import hypercoast

## Inspect sensor metadata

The registry provides one place for default variables, RGB wavelengths, supported file extensions, search/download support, and QGIS-facing metadata.

In [ ]:
registry = hypercoast.registry_as_dict()
registry["pace"]

In [ ]:
hypercoast.qgis_data_types()["Wyvern"]

## List workflow presets

In [ ]:
hypercoast.list_workflows()

## Create a small hyperspectral cube

The cube includes common coastal wavelengths used by the built-in presets.

In [ ]:
wavelengths = np.array([412.0, 555.0, 560.0, 665.0, 709.0, 860.0])
rows, cols = 30, 40
rng = np.random.default_rng(42)
base = rng.normal(0.02, 0.004, size=(wavelengths.size, rows, cols))
base[2] += 0.04  # green reflectance
base[3] += 0.015  # red reflectance
base[5] += 0.005  # NIR reflectance

dataset = xr.Dataset(
    {
        "reflectance": (
            ("wavelength", "y", "x"),
            base.astype("float32"),
        )
    },
    coords={"wavelength": wavelengths, "y": np.arange(rows), "x": np.arange(cols)},
)
dataset

## Apply workflow presets

In [ ]:
ndwi = hypercoast.apply_workflow(dataset, "ndwi")
chlorophyll = hypercoast.apply_workflow(dataset, "chlorophyll")
anomaly = hypercoast.apply_workflow(dataset, "anomaly")

xr.Dataset({"ndwi": ndwi, "chlorophyll": chlorophyll, "anomaly": anomaly})

In [ ]:
ndwi.plot.imshow(cmap="BrBG", vmin=-1, vmax=1)